# 식품 원재료 정보 Open API 수집

공공데이터포털의 `FoodRwmatrInfoService01`을 이용해 식품 원재료 정보를 수집합니다. 전체 페이지를 메모리에 모은 뒤 일반식품 원료(`LCLAS_NM == 식품원료(A코드)`)만 선별하여 CSV, JSONL, Parquet으로 저장합니다.

> 이 노트북은 **메모리 일괄 수집 방식**입니다. 실행 도중 중단되면 처음부터 다시 수집합니다. 먼저 `MODE = "sample"`로 연결을 확인한 다음 `MODE = "full"`로 변경하세요.

## 0. 실행 전 준비

인증키는 노트북에 직접 저장하지 않습니다. Jupyter를 실행하기 전에 PowerShell에서 다음과 같이 환경변수를 설정하는 방법을 권장합니다.

```powershell
$env:DATA_GO_KR_API_KEY = '공공데이터포털_일반_인증키'
jupyter lab
```

환경변수가 없으면 실행 중 숨김 입력창이 나타납니다. URL 인코딩된 키와 디코딩된 키 모두 처리하도록 구성했습니다.

In [1]:
from __future__ import annotations

from collections.abc import Callable, Mapping
from datetime import datetime
from getpass import getpass
import json
import os
from pathlib import Path
import tempfile
import time
from typing import Any
from urllib.parse import unquote

import pandas as pd
import requests

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == 'notebooks' else cwd
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = cwd

print('프로젝트 경로:', PROJECT_ROOT)

프로젝트 경로: C:\Project_AI\mle-01-final-team3


## 1. 실행 설정

처음에는 `sample`로 실행해 인증키와 응답 구조를 확인하세요. 확인 후 `full`로 바꾸면 전체 데이터를 수집합니다. `RUN_COLLECTION=True`이므로 설정 셀 이후의 수집 셀을 실행하면 실제 API를 호출합니다.

In [9]:
ENDPOINT = 'https://apis.data.go.kr/1471000/FoodRwmatrInfoService01/getFoodRwmatrList01'
API_KEY = os.getenv('DATA_GO_KR_API_KEY', '').strip()

RUN_COLLECTION = True
MODE = 'full'  # 연결 확인 후 'full'로 변경
PAGE_SIZE = 500
PAUSE_SECONDS = 0.2
REQUEST_TIMEOUT = 60
MAX_RETRIES = 4
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'food_raw_material'

if RUN_COLLECTION and not API_KEY:
    API_KEY = getpass('공공데이터포털 일반 인증키를 입력하세요: ').strip()

print({
    'mode': MODE,
    'page_size': PAGE_SIZE,
    'output_dir': str(OUTPUT_DIR),
    'api_key_set': bool(API_KEY),
})

{'mode': 'full', 'page_size': 500, 'output_dir': 'C:\\Project_AI\\mle-01-final-team3\\data\\raw\\food_raw_material', 'api_key_set': True}


## 2. 수집·필터·저장 함수

API 오류를 검사하고, 단일 객체와 목록 형태의 응답을 동일한 레코드 목록으로 정규화합니다. 네트워크 오류는 지수 대기 후 재시도합니다.

In [10]:
SUCCESS_CODES = {'0', '00', 'NORMAL_SERVICE', 'SUCCESS'}


def parse_response(payload: Mapping[str, Any]) -> tuple[int, list[dict[str, Any]]]:
    response = payload.get('response', payload)
    if not isinstance(response, Mapping):
        raise RuntimeError('API 응답의 response 형식이 예상과 다릅니다.')

    header = response.get('header') or {}
    if not isinstance(header, Mapping):
        raise RuntimeError('API 응답의 header 형식이 예상과 다릅니다.')

    result_code = str(header.get('resultCode', '00')).strip()
    result_message = str(header.get('resultMsg', 'API 오류')).strip()
    if result_code not in SUCCESS_CODES:
        raise RuntimeError(f'공공데이터포털 API 오류 {result_code}: {result_message}')

    body = response.get('body') or {}
    if not isinstance(body, Mapping):
        raise RuntimeError('API 응답의 body 형식이 예상과 다릅니다.')

    try:
        total = int(body.get('totalCount') or 0)
    except (TypeError, ValueError) as exc:
        raise RuntimeError('API 응답의 totalCount가 정수가 아닙니다.') from exc

    items = body.get('items') or {}
    if isinstance(items, Mapping):
        rows = items.get('item') or []
    elif isinstance(items, list):
        rows = items
    else:
        raise RuntimeError('API 응답의 items 형식이 예상과 다릅니다.')

    if isinstance(rows, Mapping):
        rows = [rows]
    if not isinstance(rows, list):
        raise RuntimeError('API 응답의 item 형식이 예상과 다릅니다.')

    normalized = [dict(row) for row in rows if isinstance(row, Mapping)]
    return total, normalized


def filter_general_food_materials(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        record
        for record in records
        if str(record.get('LCLAS_NM', '')).strip() == '식품원료(A코드)'
    ]


def request_page(
    endpoint: str,
    api_key: str,
    page_no: int,
    page_size: int,
    timeout: int = 60,
    retries: int = 4,
) -> Mapping[str, Any]:
    params = {
        'serviceKey': unquote(api_key.strip()),
        'pageNo': page_no,
        'numOfRows': page_size,
        'type': 'json',
    }
    last_error: Exception | None = None

    for attempt in range(retries + 1):
        try:
            response = requests.get(endpoint, params=params, timeout=timeout)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, Mapping):
                raise RuntimeError('API JSON 최상위 형식이 객체가 아닙니다.')
            return payload
        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_error = exc
            if attempt >= retries:
                break
            wait_seconds = 2 ** attempt
            print(f'요청 실패 - {wait_seconds}초 후 재시도 ({attempt + 1}/{retries})')
            time.sleep(wait_seconds)

    raise RuntimeError('식품 원재료 API 요청에 반복 실패했습니다.') from last_error


def collect_all(
    endpoint: str,
    api_key: str,
    mode: str,
    *,
    page_size: int = 999,
    pause_seconds: float = 0.2,
    timeout: int = 60,
    retries: int = 4,
    fetch_page: Callable[..., Mapping[str, Any]] = request_page,
) -> list[dict[str, Any]]:
    if not api_key.strip():
        raise ValueError('인증키가 없습니다. DATA_GO_KR_API_KEY를 설정하세요.')
    if mode not in {'sample', 'full'}:
        raise ValueError("MODE는 'sample' 또는 'full'이어야 합니다.")
    if not 1 <= page_size <= 999:
        raise ValueError('page_size는 1~999 사이여야 합니다.')

    records: list[dict[str, Any]] = []
    page_no = 1

    while True:
        payload = fetch_page(
            endpoint=endpoint,
            api_key=api_key,
            page_no=page_no,
            page_size=page_size,
            timeout=timeout,
            retries=retries,
        )
        total, rows = parse_response(payload)
        records.extend(rows)
        target = min(total, page_size) if mode == 'sample' else total
        print(f'{len(records):,}/{target:,}건 수집')

        if mode == 'sample' or not rows or len(records) >= total:
            break

        page_no += 1
        if pause_seconds:
            time.sleep(pause_seconds)

    return records


def save_outputs(
    records: list[dict[str, Any]],
    output_dir: str | Path,
    timestamp: str,
    *,
    source_endpoint: str | None = None,
    api_total_count: int | None = None,
) -> dict[str, Path]:
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    stem = f'food_raw_material_{timestamp}'
    paths = {
        'csv': destination / f'{stem}.csv',
        'jsonl': destination / f'{stem}.jsonl',
        'parquet': destination / f'{stem}.parquet',
        'metadata': destination / f'{stem}_metadata.json',
    }

    frame = pd.DataFrame(records)
    frame.to_csv(paths['csv'], index=False, encoding='utf-8-sig')
    frame.to_parquet(paths['parquet'], index=False)

    with paths['jsonl'].open('w', encoding='utf-8') as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + '\n')

    metadata = {
        'collected_at': datetime.now().astimezone().isoformat(timespec='seconds'),
        'record_count': len(records),
        'api_total_count': api_total_count,
        'columns': list(frame.columns),
        'filter': 'LCLAS_NM == "식품원료(A코드)"',
        'source_endpoint': source_endpoint,
    }
    paths['metadata'].write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return paths

## 3. 네트워크 없는 로컬 검증

실제 API 호출 전에 응답 파싱, 오류 처리, 필터링, 3페이지 수집, 파일 저장을 가짜 응답으로 검증합니다. 이 셀은 인증키를 사용하지 않습니다.

In [11]:
def _fixture_payload(total: int, rows: Any) -> dict[str, Any]:
    return {
        'response': {
            'header': {'resultCode': '00', 'resultMsg': 'NORMAL SERVICE.'},
            'body': {'totalCount': total, 'items': {'item': rows}},
        }
    }


sample_payload = _fixture_payload(
    2,
    [
        {'LCLAS_NM': '식품원료(A코드)', 'MLSFC_NM': '식물', 'RPRSNT_RAWMTRL_NM': 'Abiu열매'},
        {'LCLAS_NM': '식품첨가물(B코드)', 'MLSFC_NM': '식품첨가물', 'RPRSNT_RAWMTRL_NM': '감색소'},
    ],
)
total, parsed = parse_response(sample_payload)
assert total == 2
assert [row['RPRSNT_RAWMTRL_NM'] for row in filter_general_food_materials(parsed)] == ['Abiu열매']

calls: list[int] = []


def _fake_fetch_page(**kwargs: Any) -> dict[str, Any]:
    page_no = kwargs['page_no']
    calls.append(page_no)
    start = (page_no - 1) * 999
    count = min(999, 2001 - start)
    rows = [
        {'LCLAS_NM': '식품원료(A코드)', 'RPRSNT_RAWMTRL_NM': f'원료-{index}'}
        for index in range(start, start + count)
    ]
    return _fixture_payload(2001, rows)


fixture_records = collect_all(
    'https://example.test/api',
    'test-key',
    'full',
    page_size=999,
    pause_seconds=0,
    fetch_page=_fake_fetch_page,
)
assert len(fixture_records) == 2001
assert calls == [1, 2, 3]

with tempfile.TemporaryDirectory() as temp_dir:
    test_paths = save_outputs(
        filter_general_food_materials(parsed),
        Path(temp_dir),
        'test',
    )
    assert all(path.exists() for path in test_paths.values())

print('로컬 검증 완료: 파싱·필터·페이지 반복·파일 저장 정상')

999/2,001건 수집
1,998/2,001건 수집
2,001/2,001건 수집
로컬 검증 완료: 파싱·필터·페이지 반복·파일 저장 정상


## 4. 실제 API 수집 및 저장

`sample` 모드는 첫 페이지 최대 999건만 저장합니다. 샘플 결과를 확인한 뒤 설정 셀의 `MODE`를 `full`로 바꾸고 이 셀을 다시 실행하세요.

In [12]:
all_records: list[dict[str, Any]] = []
general_food_materials: list[dict[str, Any]] = []
saved_paths: dict[str, Path] = {}

if RUN_COLLECTION:
    all_records = collect_all(
        ENDPOINT,
        API_KEY,
        MODE,
        page_size=PAGE_SIZE,
        pause_seconds=PAUSE_SECONDS,
        timeout=REQUEST_TIMEOUT,
        retries=MAX_RETRIES,
    )
    general_food_materials = filter_general_food_materials(all_records)
    run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    saved_paths = save_outputs(
        general_food_materials,
        OUTPUT_DIR,
        run_timestamp,
        source_endpoint=ENDPOINT,
        api_total_count=len(all_records),
    )
    print(f'API 수집 레코드: {len(all_records):,}건')
    print(f'일반식품 원료: {len(general_food_materials):,}건')
    print('저장 완료:')
    for kind, path in saved_paths.items():
        print(f'  {kind}: {path}')
else:
    print('RUN_COLLECTION=False: 실제 API 수집을 건너뛰었습니다.')

500/18,650건 수집
1,000/18,650건 수집
1,500/18,650건 수집
2,000/18,650건 수집
2,500/18,650건 수집
3,000/18,650건 수집
3,500/18,650건 수집
4,000/18,650건 수집
4,500/18,650건 수집
5,000/18,650건 수집
5,500/18,650건 수집
6,000/18,650건 수집
6,500/18,650건 수집
7,000/18,650건 수집
7,500/18,650건 수집
8,000/18,650건 수집
8,500/18,650건 수집
9,000/18,650건 수집
9,500/18,650건 수집
10,000/18,650건 수집
10,500/18,650건 수집
11,000/18,650건 수집
11,500/18,650건 수집
12,000/18,650건 수집
12,500/18,650건 수집
13,000/18,650건 수집
13,500/18,650건 수집
14,000/18,650건 수집
14,500/18,650건 수집
15,000/18,650건 수집
15,500/18,650건 수집
16,000/18,650건 수집
16,500/18,650건 수집
17,000/18,650건 수집
17,500/18,650건 수집
18,000/18,650건 수집
18,500/18,650건 수집
18,650/18,650건 수집
API 수집 레코드: 18,650건
일반식품 원료: 13,880건
저장 완료:
  csv: C:\Project_AI\mle-01-final-team3\data\raw\food_raw_material\food_raw_material_20261123_015413.csv
  jsonl: C:\Project_AI\mle-01-final-team3\data\raw\food_raw_material\food_raw_material_20261123_015413.jsonl
  parquet: C:\Project_AI\mle-01-final-team3\data\raw\food_raw_material\food_raw

## 5. 결과 점검

중분류별 건수와 주요 컬럼을 확인합니다. 같은 원재료명이라도 학명·부위·상태·사용조건이 다를 수 있으므로 원재료명만으로 중복 제거하지 마세요.

In [13]:
if general_food_materials:
    result_frame = pd.DataFrame(general_food_materials)
    print('컬럼:', list(result_frame.columns))
    if 'MLSFC_NM' in result_frame.columns:
        display(result_frame['MLSFC_NM'].fillna('(없음)').value_counts().to_frame('건수'))

    preview_columns = [
        column
        for column in [
            'MLSFC_NM', 'RPRSNT_RAWMTRL_NM', 'RAWMTRL_NCKNM',
            'ENG_NM', 'SCNM', 'REGN_CD_NM', 'RAWMTRL_STATS_CD_NM',
            'USE_CND_NM',
        ]
        if column in result_frame.columns
    ]
    display(result_frame[preview_columns].head(10))
else:
    print('표시할 수집 결과가 없습니다. 먼저 실제 API 수집 셀을 실행하세요.')

컬럼: ['LCLAS_NM', 'MLSFC_NM', 'RPRSNT_RAWMTRL_NM', 'RAWMTRL_NCKNM', 'ENG_NM', 'SCNM', 'REGN_CD_NM', 'RAWMTRL_STATS_CD_NM', 'USE_CND_NM']


,건수
MLSFC_NM,
식물,10538
수산물,2288
축산물,566
미생물,182
"축,수산물 외 동물",158
기타,126
한시적인정,22


,MLSFC_NM,RPRSNT_RAWMTRL_NM,RAWMTRL_NCKNM,ENG_NM,SCNM,REGN_CD_NM,RAWMTRL_STATS_CD_NM,USE_CND_NM
0,식물,Abiu열매,None,"Yellow star apple, Caimito, Caimo, Luma",Pouteria caimito Radlk / Lucuma caimito Roem. ...,열매,None,사용가능부위 : 열매
1,식물,Abiu열매분말,None,"Yellow star apple, Caimito, Caimo, Luma",Pouteria caimito Radlk / Lucuma caimito Roem. ...,열매,None,사용가능부위 : 열매
2,식물,Agathi꽃,None,Acham,Sesbania grandiflora,꽃,None,"사용가능부위 : 잎, 꽃"
3,식물,Agathi잎,None,Acham,Sesbania grandiflora,잎,None,"사용가능부위 : 잎, 꽃"
4,식물,Alder Buckthorn,None,"Frangula bark, Frangula alnus (Mill.), Black d...",Rhamnus frangula (L),전체,None,None
5,식물,Algarrobo blanco열매,None,"Ibope, Igope, Tacu, Argentine mesquite",Prosopis alba Grisebach,열매,None,사용가능부위 : 열매
6,식물,Amaltash,None,,Cassia Fistula Linn,전체,None,None
7,식물,Ambarella열매,None,"Jobo de la India, Otaheite apple",Spondias dulcis Forst. / Spondias cytherea Sonn.,열매,None,사용가능부위 : 열매
8,식물,Anantmul잎,None,None,Hemidesmus indicus,잎,None,사용가능부위 : 잎
9,식물,Arjuna Myrobalan,None,"Arjun, Arjuna","Terminalia arjuna, Pentaptera glabra",전체,None,None


## 운영 참고사항

- 샘플 실행이 성공하면 `MODE = "full"`로 변경해 다시 실행합니다.
- 수집 결과는 `data/raw/food_raw_material/`에 생성됩니다.
- 인증 오류가 발생하면 공공데이터포털 활용신청 상태와 인증키 유효기간을 확인합니다.
- API 결과는 기본 원료정보입니다. 사용 가능·제한 조건의 법적 최종 판단은 현행 「식품의 기준 및 규격」과 식품안전나라 상세 페이지를 함께 확인합니다.
- 메모리가 부족하면 추후 페이지별 스트리밍 방식으로 변경해야 합니다.